<a href="https://colab.research.google.com/github/SydAt1/Eye_Disease_Classification/blob/main/notebooks/RandomForestClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# LOAD DATA
df = pd.read_csv("prepped_NHIS_dataset.csv")

print("Dataset shape:", df.shape)
print("\nRisk Level Distribution:")
print(df["Risk_Level"].value_counts(normalize=True).round(3))

Dataset shape: (11295, 10)

Risk Level Distribution:
Risk_Level
0    0.500
1    0.253
2    0.168
3    0.079
Name: proportion, dtype: float64


In [ ]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [ ]:
feature_cols = [c for c in df.columns if c not in ["Data_Value", "Risk_Level"]]

X = df[feature_cols]
y = df["Risk_Level"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


In [ ]:
# Trainig Random Forest Classifier

rf_clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,          # let trees grow, forest will regularize
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [ ]:
# PREDICTIONS
y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)


### Model Evaluation

In [ ]:
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
balanced_acc = balanced_accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy:       {train_acc:.4f}")
print(f"Test Accuracy:           {test_acc:.4f}")
print(f"Balanced Test Accuracy:  {balanced_acc:.4f}\n")

print("Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

Training Accuracy:       0.8373
Test Accuracy:           0.6843
Balanced Test Accuracy:  0.6080

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.83      0.74      0.78      1693
           1       0.73      0.72      0.72       857
           2       0.64      0.64      0.64       570
           3       0.19      0.33      0.24       269

    accuracy                           0.68      3389
   macro avg       0.60      0.61      0.60      3389
weighted avg       0.72      0.68      0.70      3389



In [ ]:
# Feature Importance
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_clf.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance_df.head(20).to_string(index=False))


             Feature  Importance
            Question    0.594074
           Age_range    0.135503
       RaceEthnicity    0.110472
RiskFactorResponseID    0.059663
                 Sex    0.054006
        RiskFactorID    0.034540
           YearStart    0.011742


Trying to improve model performance

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=600,
    max_depth=12,
    min_samples_leaf=4,
    # class_weight={0:1, 1:1.2, 2:1.8, 3:4}, # Accuracy with this weight was 0.76
    class_weight={0:1, 1:1.2, 2:1.5, 3:3}, #this was giving 0.78 accuracy
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

RandomForestClassifier(class_weight={0: 1, 1: 1.2, 2: 1.5, 3: 3}, max_depth=12,
                       min_samples_leaf=4, n_estimators=600, n_jobs=-1,
                       random_state=42)

In [ ]:
# PREDICTIONS
y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)

In [ ]:
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy:       {train_acc:.4f}")
print(f"Test Accuracy:           {test_acc:.4f}")

print("Classification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

Training Accuracy:       0.8334
Test Accuracy:           0.7816
Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.82      0.94      0.87      1693
           1       0.72      0.75      0.74       857
           2       0.79      0.58      0.67       570
           3       0.61      0.31      0.41       269

    accuracy                           0.78      3389
   macro avg       0.74      0.65      0.67      3389
weighted avg       0.77      0.78      0.77      3389

